# Cover Letter Agent - V1

A deliberately simple LangGraph pipeline. Every node has one responsibility, and the graph
decides what happens next based on state.

```
                         START
                           │
                           ▼
                    analyze_job            job ad  -> structured requirements
                           │
                           ▼
                     analyze_cv            CV      -> candidate profile (source of truth)
                           │
                           ▼
                    match_evidence         requirement -> evidence + confidence
                           │
                           ▼
                     write_draft           writes only from matched evidence
                           │
                           ▼
                ┌──► validate_factuality   does the draft claim anything unsupported?
                │          │
                │    ┌─────┴─────┐
                │  PASS        FAIL
                │    │           │
                │    ▼           │
                │ quality_check  │         is it actually a good letter?
                │    │           │
                │ ┌──┴───┐       │
                │ GOOD  WEAK     │
                │ │      │       │
                │ │      ▼       ▼
                └─┼──── revise ◄─┘
                  │
                  ▼
              finalize
                  │
                  ▼
                 END
```

## The idea

The model never receives one giant prompt asking it to do everything. Messy text becomes
structured information *before* anything is written, and the writer is handed
`requirement → verified evidence` rather than `CV + job ad`.

Two independent gates guard the output, because they catch different failures:

- **`validate_factuality`** catches invention. A letter claiming Docker experience you do not
  have is disqualifying in a way that bad prose is not.
- **`quality_check`** catches blandness. *"I know Python. I have worked as a data scientist. I am
  interested in your company."* is perfectly factual and a terrible letter.

`revise` serves both, so it receives one of two kinds of feedback: *you invented something*, or
*this paragraph is generic*. After either kind it returns to `validate_factuality` - because a
revision made for style can introduce a claim that was never true.

---
## 1. Setup

```bash
uv venv --python 3.13
uv pip install langgraph langchain-anthropic python-dotenv ipykernel pypdf python-docx
```

`.env` next to this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
```

In [ ]:
import json
import os
import re
from dataclasses import dataclass, asdict
from datetime import date
from pathlib import Path
from typing import Literal, Optional, TypedDict

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

load_dotenv(find_dotenv(usecwd=True))
assert os.environ.get("ANTHROPIC_API_KEY"), "Put ANTHROPIC_API_KEY in a .env file next to this notebook"
print("Anthropic key loaded.")

---
## 2. Configuration

The two thresholds are the interesting settings. `factuality_threshold` is high on purpose: an
invented claim is a worse failure than a dull sentence, so that gate should be hard to pass.

In [ ]:
@dataclass
class Config:
    model: str = "claude-opus-5"

    # gate 1 - factual grounding
    factuality_threshold: float = 0.90

    # gate 2 - quality, all three must clear
    relevance_threshold: float = 0.80
    specificity_threshold: float = 0.75
    writing_threshold: float = 0.80

    max_revisions: int = 2          # without this an agent can loop indefinitely

    target_words: int = 350
    max_words: int = 420

    inputs_dir: Path = Path("inputs")
    outputs_dir: Path = Path("outputs")


cfg = Config()
cfg.outputs_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps({k: str(v) for k, v in asdict(cfg).items()}, indent=2))

---
## 3. Inputs

Two files: the job ad and your CV.

```
inputs/
├── job_posting.txt     # or .pdf / .docx / .md
└── cv.txt              # or .pdf / .docx
```

In [ ]:
def read_document(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in {".txt", ".md"}:
        return path.read_text(encoding="utf-8")
    if suffix == ".pdf":
        from pypdf import PdfReader
        return "\n".join((page.extract_text() or "") for page in PdfReader(str(path)).pages)
    if suffix == ".docx":
        import docx
        return "\n".join(p.text for p in docx.Document(str(path)).paragraphs)
    raise ValueError(f"Unsupported file type: {path.name}")


def find(stem: str) -> Path:
    matches = [p for p in cfg.inputs_dir.glob(f"{stem}.*")
               if p.suffix.lower() in {".txt", ".md", ".pdf", ".docx"}]
    if not matches:
        raise FileNotFoundError(f"No {stem}.(txt|md|pdf|docx) in {cfg.inputs_dir}/")
    return matches[0]


inputs = {"job_ad": read_document(find("job_posting")), "cv": read_document(find("cv"))}
print(f"job ad : {len(inputs['job_ad'].split())} words")
print(f"cv     : {len(inputs['cv'].split())} words")

---
## 4. Schemas, state and helpers

Each analytical node returns a validated object rather than prose. That is what lets the graph
branch on the result - you cannot write `if factuality_score < threshold` against a paragraph of
English.

In [ ]:
# ---------- analyze_job ----------
class Requirement(BaseModel):
    requirement: str
    importance: Literal["required", "preferred"]


class JobRequirements(BaseModel):
    role_title: str
    company: str
    requirements: list[Requirement]


# ---------- analyze_cv ----------
class ExperienceItem(BaseModel):
    role: str
    organization: str
    years: float = Field(description="duration in years, e.g. 1.5")
    achievements: list[str] = Field(description="what they actually did, quoted from the CV")


class CandidateProfile(BaseModel):
    full_name: str
    skills: list[str]
    experience: list[ExperienceItem]
    education: list[str]


# ---------- match_evidence ----------
class EvidenceItem(BaseModel):
    requirement: str
    evidence: Optional[str] = Field(description="the specific CV fact supporting it, or null if none")
    confidence: float = Field(ge=0.0, le=1.0)


class EvidenceMap(BaseModel):
    items: list[EvidenceItem]


# ---------- the two gates ----------
class FactualityReport(BaseModel):
    unsupported_claims: list[str] = Field(description="claims in the draft with no basis in the profile")
    factuality_score: float = Field(ge=0.0, le=1.0)


class QualityReport(BaseModel):
    relevance_score: float = Field(ge=0.0, le=1.0)
    specificity_score: float = Field(ge=0.0, le=1.0)
    writing_score: float = Field(ge=0.0, le=1.0)
    issues: list[str] = Field(description="specific, applicable fixes in priority order")


class State(TypedDict, total=False):
    job_ad: str
    cv: str

    job_requirements: list
    role_title: str
    company: str
    candidate_profile: dict
    evidence_map: list

    draft: str
    unsupported_claims: list
    factuality_score: float
    quality_scores: dict
    quality_issues: list

    revisions: int
    trace: list
    warnings: list
    final_letter: str


def ask(schema, system: str, user: str, attempts: int = 3, max_tokens: int = 16000):
    """A structured call, retried on a malformed response.

    Keep max_tokens generous - Claude Opus 5 thinks by default and those tokens come out of the
    same budget, so starving it truncates the tool call rather than the prose."""
    chain = ChatAnthropic(model=cfg.model, max_tokens=max_tokens).with_structured_output(schema)
    messages = [SystemMessage(content=system), HumanMessage(content=user)]
    for attempt in range(1, attempts + 1):
        try:
            return chain.invoke(messages)
        except Exception as exc:
            if attempt == attempts:
                raise
            print(f"    ({schema.__name__} attempt {attempt} failed: {type(exc).__name__}, retrying)")


def prose(system: str, user: str, max_tokens: int = 12000) -> str:
    """A free-text call, guarding against an empty completion."""
    message = ChatAnthropic(model=cfg.model, max_tokens=max_tokens).invoke(
        [SystemMessage(content=system), HumanMessage(content=user)])
    text = message.text if isinstance(message.text, str) else message.text()
    if not text.strip():
        raise RuntimeError("Model returned no text - thinking consumed the budget; raise max_tokens.")
    return text.strip()


print("schemas, state and helpers defined")

---
## 5. `analyze_job`

Messy job ad → structured requirements, each marked `required` or `preferred`.

The distinction matters downstream: a missing `required` item is a hole the letter has to work
around, while a missing `preferred` one is usually best left unmentioned.

In [ ]:
JOB_SYSTEM = """Extract the requirements from a job advertisement.

- One entry per distinct requirement. Split compound bullets: "Python and SQL" is two entries.
- `importance` is "required" when the ad states it as necessary (must, required, you have), and
  "preferred" when it is framed as a bonus (nice to have, plus, ideally).
- Include technical skills, tools, domain experience, qualifications and soft requirements.
- Ignore the benefits and perks section entirely - those are what the company offers, not asks.
- If the ad is not in English, extract in English anyway."""


def analyze_job(state: State) -> dict:
    result = ask(JobRequirements, JOB_SYSTEM, f"Job advertisement:\n\n{state['job_ad']}")
    required = [r for r in result.requirements if r.importance == "required"]

    print(f"[analyze_job] {result.company} - {result.role_title}")
    print(f"[analyze_job] {len(result.requirements)} requirements "
          f"({len(required)} required, {len(result.requirements) - len(required)} preferred)")

    return {
        "job_requirements": [r.model_dump() for r in result.requirements],
        "role_title": result.role_title,
        "company": result.company,
        "trace": ["analyze_job"],
    }

---
## 6. `analyze_cv`

The CV becomes a structured profile, and that profile is the system's **source of truth**. From
here on, nothing downstream reads the raw CV.

That is the point. If the writer can only see extracted facts, it has much less room to invent -
and the fact checker later has something concrete to check against.

In [ ]:
CV_SYSTEM = """Extract a factual profile from a CV.

- `skills`: technologies, methods and tools the CV actually names. Do not infer skills that
  "obviously" follow from a role - if the CV does not say SQL, SQL does not go in the list.
- `experience`: one entry per role. `achievements` are what the person did, stated plainly and
  closely following the CV's own wording. Keep any numbers exactly as written.
- `years`: compute from the stated dates. Use decimals for part-years.
- Add nothing that is not on the page. This profile is the source of truth for a later fact
  check, so an invention here defeats the whole system."""


def analyze_cv(state: State) -> dict:
    profile = ask(CandidateProfile, CV_SYSTEM, f"CV:\n\n{state['cv']}")
    print(f"[analyze_cv] {profile.full_name}: {len(profile.skills)} skills, "
          f"{len(profile.experience)} roles, {len(profile.education)} qualifications")
    return {"candidate_profile": profile.model_dump(), "trace": state["trace"] + ["analyze_cv"]}


def render_profile(profile: dict) -> str:
    lines = [f"Name: {profile['full_name']}", f"Skills: {', '.join(profile['skills'])}", ""]
    for job in profile["experience"]:
        lines.append(f"- {job['role']} at {job['organization']} ({job['years']} years)")
        lines += [f"    * {a}" for a in job["achievements"]]
    lines += ["", "Education: " + "; ".join(profile["education"])]
    return "\n".join(lines)

---
## 7. `match_evidence`

The node that makes the rest work.

For every requirement it finds the specific CV fact that supports it, with a confidence score -
or records `null` and `0.0`. An honest zero is the most valuable output here: it is what tells
the writer to stay quiet about Docker rather than to improvise.

In [ ]:
MATCH_SYSTEM = """Match each job requirement against a candidate's profile.

For every requirement return:
- `evidence`: the specific fact from the profile that supports it, quoted or closely paraphrased.
  Use null when nothing in the profile supports it.
- `confidence`: 0.0 to 1.0.
    1.0    the profile states it outright
    0.7-0.9 strongly implied by something concrete
    0.3-0.6 adjacent or transferable, would need framing
    0.0    nothing supports it

Be strict. Marking a gap as partial produces a letter that collapses at interview, and a
confident zero is far more useful to the writer than a hopeful 0.4. Never combine unrelated
facts into evidence that the profile does not actually contain.

Return one entry per requirement, in the order given."""


def match_evidence(state: State) -> dict:
    requirements = "\n".join(
        f"- [{r['importance']}] {r['requirement']}" for r in state["job_requirements"])

    result = ask(EvidenceMap, MATCH_SYSTEM,
                 f"JOB REQUIREMENTS:\n{requirements}\n\n"
                 f"CANDIDATE PROFILE:\n{render_profile(state['candidate_profile'])}")

    items = [i.model_dump() for i in result.items]
    strong = [i for i in items if i["confidence"] >= 0.7]
    none = [i for i in items if i["confidence"] == 0.0]

    print(f"[match_evidence] {len(strong)} strong, "
          f"{len(items) - len(strong) - len(none)} partial, {len(none)} with no evidence")
    for item in none:
        print(f"    no evidence: {item['requirement']}")

    return {"evidence_map": items, "trace": state["trace"] + ["match_evidence"]}

---
## 8. `write_draft`

The writer receives requirements and matched evidence - never the raw CV.

Requirements with no evidence are shown explicitly as `NO EVIDENCE`. Listing them is safer than
omitting them: a silent gap invites the model to fill it, while a labelled one is an instruction
to leave it alone.

In [ ]:
WRITE_SYSTEM = """You write a cover letter from verified evidence.

You may only make claims that appear in the evidence below. This is absolute:
- Evidence marked NO EVIDENCE means the candidate has not demonstrated it. Do not claim it, do
  not imply it, do not say they are "eager to develop" it unless asked. Stay silent.
- Invent no metric, date, tool or responsibility that is not in the evidence.
- Do not inflate. "Familiar with X" must not become "extensive experience with X".

Emphasise the requirements with the strongest evidence, weighted toward those marked
[required]. Build the letter around two or three of them rather than listing everything.

Structure: a short opening on why this role, two or three body paragraphs of evidence, and a
brief closing. Around {target} words, never more than {maximum}.

Style:
- Open with substance. Never "I am excited to apply for" or "I am writing to apply for".
- Every claim carries its evidence in the same sentence. No adjective stands alone as a
  qualification.
- Concrete nouns and verbs. If a sentence would survive swapping in a different candidate, cut it.

Output the letter only, salutation through sign-off. No preamble, no commentary."""


def render_evidence(evidence_map: list, requirements: list) -> str:
    importance = {r["requirement"]: r["importance"] for r in requirements}
    lines = []
    for item in sorted(evidence_map, key=lambda i: -i["confidence"]):
        tag = importance.get(item["requirement"], "required")
        if item["evidence"] and item["confidence"] > 0:
            lines.append(f"[{tag}] {item['requirement']} (confidence {item['confidence']:.2f})\n"
                         f"    -> {item['evidence']}")
        else:
            lines.append(f"[{tag}] {item['requirement']}\n    -> NO EVIDENCE")
    return "\n".join(lines)


def write_draft(state: State) -> dict:
    text = prose(
        WRITE_SYSTEM.format(target=cfg.target_words, maximum=cfg.max_words),
        f"Candidate: {state['candidate_profile']['full_name']}\n"
        f"Applying for: {state['role_title']} at {state['company']}\n\n"
        f"REQUIREMENT -> EVIDENCE:\n{render_evidence(state['evidence_map'], state['job_requirements'])}")

    print(f"[write_draft] {len(text.split())} words")
    return {"draft": text, "revisions": 0, "trace": state["trace"] + ["write_draft"]}

---
## 9. `validate_factuality` - gate 1

The fact checker. It sees the profile, the evidence map and the draft, and asks one question:
does the letter claim anything the candidate cannot support?

Note it checks against the **profile**, not the original CV. The profile is what the writer was
allowed to use, so it is the right standard - and it keeps the check narrow enough to be
reliable.

In [ ]:
FACTUALITY_SYSTEM = """You fact-check a cover letter against a candidate's verified profile.

List every claim in the letter that the profile does not support. A claim is unsupported when:
- the profile contains nothing about it
- it overstates what the profile says ("extensive experience" where the profile shows one project)
- it invents a number, date, tool or responsibility
- it implies a capability by association that the profile does not establish

Quote the offending phrase so it can be found and fixed.

Not unsupported: motivation, interest, stated intent, ordinary connective prose, or a fair
paraphrase of something the profile does contain.

`factuality_score`: 1.0 when every claim is supported. Deduct in proportion to how serious and
how central each unsupported claim is - an invented technical skill matters far more than a mild
adjective. Be strict; this gate exists to catch exactly this."""


def validate_factuality(state: State) -> dict:
    report = ask(FactualityReport, FACTUALITY_SYSTEM,
                 f"CANDIDATE PROFILE (the source of truth):\n"
                 f"{render_profile(state['candidate_profile'])}\n\n"
                 f"EVIDENCE THE WRITER WAS GIVEN:\n"
                 f"{render_evidence(state['evidence_map'], state['job_requirements'])}\n\n"
                 f"THE LETTER:\n---\n{state['draft']}\n---")

    print(f"[validate_factuality] score {report.factuality_score:.2f} "
          f"(threshold {cfg.factuality_threshold}), "
          f"{len(report.unsupported_claims)} unsupported claim(s)")
    for claim in report.unsupported_claims:
        print(f"    ! {claim}")

    return {"unsupported_claims": report.unsupported_claims,
            "factuality_score": report.factuality_score,
            "trace": state["trace"] + [f"validate_factuality({report.factuality_score:.2f})"]}

---
## 10. `quality_check` - gate 2

Passing the fact check says the letter is *true*. It says nothing about whether it is any good.

Three scores, all of which must clear their threshold. The `issues` list is what `revise`
actually consumes, so each one has to name a change rather than a complaint.

In [ ]:
QUALITY_SYSTEM = """You assess the quality of a cover letter. Score 0.0-1.0.

- relevance_score: does it address what this job actually asks for, weighted toward the
  [required] items? A letter dwelling on unrequested strengths scores low.
- specificity_score: named methods, tools and outcomes against vague self-description. "Strong
  analytical skills" is near zero. A named method with a stated result is near one.
- writing_score: is it well written? Penalise generic openings, throat-clearing, repetition,
  filler, and any sentence that would survive swapping in a different candidate.

`issues`: the changes that would raise the scores most, in priority order. Each must name what to
change and what to change it to. "Be more specific" is not an issue; "replace 'improved model
performance' in paragraph 2 with the outperformed-baselines detail from the evidence" is.

Judge only what is here. Do not suggest adding claims the evidence does not support - that is the
other gate's job, and suggesting it would put the two gates in conflict."""


def quality_check(state: State) -> dict:
    report = ask(QualityReport, QUALITY_SYSTEM,
                 f"JOB: {state['role_title']} at {state['company']}\n\n"
                 f"REQUIREMENT -> EVIDENCE:\n"
                 f"{render_evidence(state['evidence_map'], state['job_requirements'])}\n\n"
                 f"THE LETTER:\n---\n{state['draft']}\n---")

    scores = {"relevance": report.relevance_score,
              "specificity": report.specificity_score,
              "writing": report.writing_score}

    print(f"[quality_check] relevance {scores['relevance']:.2f} "
          f"specificity {scores['specificity']:.2f} writing {scores['writing']:.2f}")
    for issue in report.issues[:4]:
        print(f"    - {issue}")

    return {"quality_scores": scores, "quality_issues": report.issues,
            "trace": state["trace"] + ["quality_check"]}

---
## 11. `revise` and the two routers

`revise` serves both gates, so it first works out which problem it is solving: a failed fact
check takes priority over a quality complaint, because a true dull letter beats a polished false
one.

Both routers respect `max_revisions`. Without that ceiling the graph can loop indefinitely - two
gates that disagree will happily trade a draft back and forth forever.

One deliberate choice: `revise` always returns to `validate_factuality`, never straight to
`quality_check`. A revision made purely for style can introduce a claim that was never true, so
every new draft is re-checked from the top.

In [ ]:
REVISE_SYSTEM = """You revise a cover letter by applying specific fixes.

Change what the problems identify and leave the rest alone. This is editing, not rewriting: a
paragraph nobody complained about should come back recognisably intact.

Introduce no new claims. You may only use the evidence supplied - if a fix seems to need a fact
that is not there, cut the sentence instead of inventing support for it.

Output the revised letter only."""


def revise(state: State) -> dict:
    if state["factuality_score"] < cfg.factuality_threshold:
        reason = "factuality"
        problems = [f"UNSUPPORTED CLAIM - remove or rewrite: {c}" for c in state["unsupported_claims"]]
    else:
        reason = "quality"
        problems = state.get("quality_issues", [])

    revisions = state.get("revisions", 0) + 1
    text = prose(
        REVISE_SYSTEM + "\n\n" + WRITE_SYSTEM.format(target=cfg.target_words, maximum=cfg.max_words),
        f"CURRENT LETTER:\n---\n{state['draft']}\n---\n\n"
        f"PROBLEMS TO FIX ({reason}):\n" + "\n".join(f"- {p}" for p in problems) + "\n\n"
        f"THE ONLY EVIDENCE YOU MAY USE:\n"
        f"{render_evidence(state['evidence_map'], state['job_requirements'])}")

    print(f"[revise] pass {revisions}/{cfg.max_revisions} on {reason} -> {len(text.split())} words")
    return {"draft": text, "revisions": revisions,
            "trace": state["trace"] + [f"revise({reason})"]}


def route_factuality(state: State) -> Literal["quality_check", "revise", "finalize"]:
    if state["factuality_score"] >= cfg.factuality_threshold:
        print("[route] factuality PASS -> quality_check")
        return "quality_check"
    if state.get("revisions", 0) >= cfg.max_revisions:
        print("[route] factuality FAIL but revision budget spent -> finalize (flagged)")
        return "finalize"
    print("[route] factuality FAIL -> revise")
    return "revise"


def route_quality(state: State) -> Literal["finalize", "revise"]:
    scores = state["quality_scores"]
    weak = [name for name, threshold in
            [("relevance", cfg.relevance_threshold),
             ("specificity", cfg.specificity_threshold),
             ("writing", cfg.writing_threshold)]
            if scores[name] < threshold]

    if not weak:
        print("[route] quality GOOD -> finalize")
        return "finalize"
    if state.get("revisions", 0) >= cfg.max_revisions:
        print(f"[route] quality WEAK ({', '.join(weak)}) but budget spent -> finalize")
        return "finalize"
    print(f"[route] quality WEAK ({', '.join(weak)}) -> revise")
    return "revise"

---
## 12. `finalize`

A boring node, which is the point - the difficult work has already happened. No model call:
it tidies formatting, records anything that failed to clear a gate, and writes the file.

The warnings matter. A letter can arrive here with unresolved problems because the revision
budget ran out, and finishing quietly in that case would hide exactly what you need to know.

In [ ]:
def finalize(state: State) -> dict:
    text = state["draft"]

    # strip anything the model may have wrapped around the letter
    text = re.sub(r"^```[a-z]*\n|\n```$", "", text.strip())
    text = re.sub(r"^#{1,6} .*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    warnings = []
    if state.get("factuality_score", 1.0) < cfg.factuality_threshold:
        warnings.append(
            f"factuality {state['factuality_score']:.2f} below {cfg.factuality_threshold} - "
            f"unresolved: {'; '.join(state.get('unsupported_claims', []))}")

    for name, threshold in [("relevance", cfg.relevance_threshold),
                            ("specificity", cfg.specificity_threshold),
                            ("writing", cfg.writing_threshold)]:
        score = state.get("quality_scores", {}).get(name)
        if score is not None and score < threshold:
            warnings.append(f"{name} {score:.2f} below {threshold}")

    words = len(text.split())
    if words > cfg.max_words:
        warnings.append(f"{words} words, over the {cfg.max_words} limit")

    slug = re.sub(r"[^a-z0-9]+", "-", f"{state['company']}-{state['role_title']}".lower()).strip("-")[:60]
    path = cfg.outputs_dir / f"{date.today().isoformat()}_{slug}_v1.md"
    path.write_text(text, encoding="utf-8")

    print(f"[finalize] {words} words -> {path}")
    for warning in warnings:
        print(f"    WARNING: {warning}")

    return {"final_letter": text, "warnings": warnings,
            "trace": state["trace"] + ["finalize"]}

---
## 13. Assemble the graph

In [ ]:
builder = StateGraph(State)

for name, fn in [("analyze_job", analyze_job), ("analyze_cv", analyze_cv),
                 ("match_evidence", match_evidence), ("write_draft", write_draft),
                 ("validate_factuality", validate_factuality), ("quality_check", quality_check),
                 ("revise", revise), ("finalize", finalize)]:
    builder.add_node(name, fn)

builder.add_edge(START, "analyze_job")
builder.add_edge("analyze_job", "analyze_cv")
builder.add_edge("analyze_cv", "match_evidence")
builder.add_edge("match_evidence", "write_draft")
builder.add_edge("write_draft", "validate_factuality")

builder.add_conditional_edges("validate_factuality", route_factuality,
                              {"quality_check": "quality_check",
                               "revise": "revise",
                               "finalize": "finalize"})

builder.add_conditional_edges("quality_check", route_quality,
                              {"finalize": "finalize", "revise": "revise"})

# every revision is re-checked from the top, including one made only for style
builder.add_edge("revise", "validate_factuality")
builder.add_edge("finalize", END)

graph = builder.compile()
print("graph compiled")

In [ ]:
from IPython.display import Image

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:                     # the PNG comes from a remote renderer
    print(f"(diagram unavailable: {exc})\n")
    print(graph.get_graph().draw_mermaid())

---
## 14. Run it

No interrupt in V1 - it runs straight through. Around six to nine model calls depending on how
many revisions the gates demand.

In [ ]:
result = graph.invoke(inputs, {"recursion_limit": 40})
print("\n" + "=" * 70)
print(" -> ".join(result["trace"]))

In [ ]:
print(f"factuality : {result['factuality_score']:.2f}  (threshold {cfg.factuality_threshold})")
for name, score in result.get("quality_scores", {}).items():
    print(f"{name:<11}: {score:.2f}")
print(f"revisions  : {result.get('revisions', 0)}")

if result.get("warnings"):
    print("\nFINISHED WITH WARNINGS")
    for warning in result["warnings"]:
        print("  ! " + warning)

display(Markdown("---\n## Final letter\n\n" + result["final_letter"]))

---
## 15. Inspecting the intermediate state

The structured stages are the useful part when something goes wrong. If the letter is bland, the
evidence map usually explains why before the prose does.

In [ ]:
print("REQUIREMENT -> EVIDENCE\n")
for item in sorted(result["evidence_map"], key=lambda i: -i["confidence"]):
    mark = "OK  " if item["confidence"] >= 0.7 else ("~   " if item["confidence"] > 0 else "NONE")
    print(f"{mark} {item['confidence']:.2f}  {item['requirement']}")
    if item["evidence"]:
        print(f"            {item['evidence'][:100]}")

---
## What is deliberately not here

This is V1. Four things were left out on purpose, and each is a reasonable next step:

**Human review.** The graph runs start to finish. Adding an `interrupt()` before `finalize` would
let you approve or send feedback.

**Company research.** Nothing here knows anything about the employer beyond what the ad says, so
the opening paragraph can only be as specific as the ad itself.

**Employer anonymisation.** This version names every organisation on your CV. If you need some of
them hidden, that has to be added before you send anything it produces.

**Memory.** Every run starts cold. Nothing is cached between applications, so the CV is
re-analysed each time even though it rarely changes.

## Tuning the gates

`factuality_threshold` at 0.90 is strict. If runs spend both revisions fighting over mild
adjectives, either lower it or sharpen the fact-checker's definition of "unsupported" - the
prompt already tells it that motivation and connective prose do not count, and that boundary is
the first thing to adjust.

If a run ends with `quality WEAK but budget spent`, raising `max_revisions` is the obvious move,
but check the evidence map first. Weak quality with a thin evidence map is not a writing problem,
and more revisions will not fix it.